In [55]:
import urllib.request
import pandas as pd
import subprocess
import numpy as np
import h5py

from scipy.sparse import csc_matrix

data_root="/home/mcn26/palmer_scratch/tabula_data"

In [56]:
dat=pd.read_csv(f"{data_root}/seelig_mpra_unpro.tsv",index_col=0,sep="\t")

In [57]:
dat

,Gene Name,Gene,A9_A2_A2,A6_A2_A2,A2_B1_A2,A5_B2_A2,A1_B2_A2,A7_B3_A2,A4_A1_A2,A5_A1_A2,...,A3_F2_F8,A5_F4_F8,A9_F4_F8,A6_F5_F8,A4_F6_F8,A7_F6_F8,A12_F7_F8,A5_F7_F8,A5_F8_F8,A6_F8_F8
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AAAATTAAACACTCGGGACTTGTCCGGGCATGCTGGCTGACTTGGC...,AAAATTAAACACTCGGGACTTGTCCGGGCATGCTGGCTGACTTGGC...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
2,AAAATTAGCCGGGCGTGGTAGCAGGCGCCTGTAGTCCCAGCTACTC...,AAAATTAGCCGGGCGTGGTAGCAGGCGCCTGTAGTCCCAGCTACTC...,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AAACAGGTCGGGGGTTAATCCATACACACGCTGGGGTTTTGCCCAG...,AAACAGGTCGGGGGTTAATCCATACACACGCTGGGGTTTTGCCCAG...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,AAACATTCATGTCAGGGCATGTGGGCTTGTAACTTTGAACCCCTGC...,AAACATTCATGTCAGGGCATGTGGGCTTGTAACTTTGAACCCCTGC...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1340,TTTTTCTTACGCGGGTCATCACTCGTATGAAATGACTCACGCGACT...,TTTTTCTTACGCGGGTCATCACTCGTATGAAATGACTCACGCGACT...,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1341,TTTTTGTTGGCGCGCGCGCCTGAAGCGGGACTGCCAGGTGGCGCGC...,TTTTTGTTGGCGCGCGCGCCTGAAGCGGGACTGCCAGGTGGCGCGC...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1342,TTTTTGTTTGACCCCTGTAATGTTTGTTCCCAGGGAACATGCCGGG...,TTTTTGTTTGACCCCTGTAATGTTTGTTCCCAGGGAACATGCCGGG...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1343,TTTTTGTTTGTACAAAGCTCTGTTTGACCCCTGCGGGCATGCCGGG...,TTTTTGTTTGTACAAAGCTCTGTTTGACCCCTGCGGGCATGCCGGG...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [58]:
dat.index=dat['Gene']
dat.drop(['Gene Name','Gene'],axis=1,inplace=True)

In [59]:
dat=dat.stack().reset_index()

In [60]:
dat.rename({'level_1':'cell_bc',0:'umis'},axis=1,inplace=True)

In [61]:
dat['umis']=dat['umis'].astype(int)

In [62]:
dat

,Gene,cell_bc,umis
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A9_A2_A2,0
1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A6_A2_A2,0
2,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A2_B1_A2,0
3,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A5_B2_A2,0
4,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A1_B2_A2,0
...,...,...,...
14310795,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A7_F6_F8,0
14310796,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A12_F7_F8,0
14310797,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F7_F8,0
14310798,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F8_F8,0


In [63]:
acceptible_cells_and_cell_type=pd.read_csv("acceptible_cells_and_cell_type.tsv",sep="\t")

In [64]:
acceptible_cells_and_cell_type

,barcode,cell_type
0,A9_A2_A2,HEPG2
1,A6_A2_A2,K562
2,A2_B1_A2,HEPG2
3,A5_B2_A2,K562
4,A1_B2_A2,HEPG2
...,...,...
9722,A7_F6_F8,K562
9723,A12_F7_F8,HEPG2
9724,A5_F7_F8,K562
9725,A5_F8_F8,K562


In [65]:
acceptible_cells_and_cell_type.rename({'barcode':'cell_bc'},axis=1,inplace=True)

In [66]:
#make sure no dup cell barcodes : would cause duplicated data
assert len(acceptible_cells_and_cell_type["cell_bc"])==len(acceptible_cells_and_cell_type["cell_bc"].unique())

In [67]:
merged=dat.merge(acceptible_cells_and_cell_type,how="inner",on="cell_bc")

In [68]:
print(f'retained={len(merged)/len(dat)*100}%.')

retained=91.41917293233082%.


In [69]:
merged

,Gene,cell_bc,umis,cell_type
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A9_A2_A2,0,HEPG2
1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A6_A2_A2,0,K562
2,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A2_B1_A2,0,HEPG2
3,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A5_B2_A2,0,K562
4,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A1_B2_A2,0,HEPG2
...,...,...,...,...
13082810,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A7_F6_F8,0,K562
13082811,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A12_F7_F8,0,HEPG2
13082812,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F7_F8,0,K562
13082813,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F8_F8,0,K562


In [73]:
merged.rename({'Gene':'cre_id'},axis=1,inplace=True)

In [74]:
merged['rep_id'] = np.nan
merged

,cre_id,cell_bc,umis,cell_type,rep_id
0,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A9_A2_A2,0,HEPG2,NaN
1,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A6_A2_A2,0,K562,NaN
2,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A2_B1_A2,0,HEPG2,NaN
3,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A5_B2_A2,0,K562,NaN
4,AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...,A1_B2_A2,0,HEPG2,NaN
...,...,...,...,...,...
13082810,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A7_F6_F8,0,K562,NaN
13082811,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A12_F7_F8,0,HEPG2,NaN
13082812,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F7_F8,0,K562,NaN
13082813,TTTTTTGTTTTGTTCAATGTTGAACATTCTACCGGGATCATTGTTT...,A5_F8_F8,0,K562,NaN


In [76]:
merged[['cell_bc','rep_id','cre_id','cell_type','umis']].to_csv(f'{data_root}/seelig.tsv',sep='\t')